# 03a - Thermostat Custom Environment

## Block 4: Your First Custom Environment

Up to this point, we have relied on pre-built environments provided by Gymnasium. However, in the real world, you will almost always need to build your own custom environments to model your specific business problems, robots, or games.

For this final challenge, you will choose **one of the two custom environments below** to implement and solve.

**Your Mission:**
1. Read the specifications for both environments.
2. Choose *one* that interests you.
3. Analyze its `observation_space` and `action_space`.
4. Based on whether the spaces are discrete or continuous, decide which algorithm we covered today (**Q-Learning** or **REINFORCE**) is the correct tool for the job.
5. Train an agent to solve it!

---

### Option A: The Smart Thermostat (`DiscreteThermostatEnv`)



In this environment, your agent is a smart home thermostat trying to reach a comfortable target temperature as quickly as possible.

| Feature | Description |
| :--- | :--- |
| **Observation Space** | `Discrete(21)`: The current temperature, ranging from 0 to 20. |
| **Action Space** | `Discrete(3)`: <br>`0`: Cool down (-1)<br>`1`: Heat up (+1)<br>`2`: Idle (Do nothing) |
| **Goal** | Reach the `target_temp` of 10. |
| **Rewards** | `0` if exactly at the target temperature. <br> `-1` for every unit of temperature away from the target (penalty for discomfort). |
| **Termination / Truncation** | Episode ends instantly in success if the target is reached, or truncates after `20` steps. |

---

### Challenge Instructions

Below, you will find the raw Python code defining both classes.

1. Run the cell to register the classes in your notebook.
2. Instantiate your chosen environment using `env = DiscreteThermostatEnv()`
3. Build the training loop! *Hint: Look back at Block 2 and Block 3. Which framework aligns with the math of your chosen environment?*

In [ ]:
# Copy here the correct Agent!

In [ ]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np

class DiscreteThermostatEnv(gym.Env):
    metadata = {"render_modes": ["human"], "render_fps": 4}

    def __init__(self, render_mode="human"):
        super().__init__()
        self.render_mode = render_mode

        # TODO: State: Discrete temperatures from 0 to 20
        self.observation_space = ...
        # TODO: Actions: 0 (Cool -1), 1 (Heat +1), 2 (Off 0)
        self.action_space = ...

        self.target_temp = 10
        self.max_steps = 20
        self.current_step = 0
        self.state = 0

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.current_step = 0
        # TODO: Start at a random discrete temperature
        self.state = ...
        return self.state, {}

    def step(self, action):
        self.current_step += 1

        # Apply actions
        if action == 0:
            self.state = ...
        elif action == 1:
            self.state = ...
        # if action == 2, do nothing

        # TODO: Clip to ensure it stays within our discrete observation space boundaries (Hint: np.clip)
        self.state = ...

        # TODO: Reward: 0 if at target, -1 * every step away
        reward = ...

        # Terminate if we perfectly hit the target, truncate if we run out of time
        terminated = bool(self.state == self.target_temp)
        truncated = self.current_step >= self.max_steps

        return self.state, float(reward), terminated, truncated, {}

    def render(self):
        if self.render_mode == "human":
            # Text-based visualizer
            bar = ["-"] * 21
            bar[self.target_temp] = "T" # Target
            if self.state == self.target_temp:
                bar[self.state] = "🌟" # Perfect!
            else:
                bar[self.state] = "O"  # Current Temp

            visual = "".join(bar)
            print(f"Step: {self.current_step:02d} | Thermostat: [{visual}] | Temp: {self.state} | Target: {self.target_temp}")

    def close(self):
        pass

In [ ]:
from tqdm import tqdm  # Progress bar

# Training hyperparameters
learning_rate = 0.01        # How fast to learn (higher = faster but less stable)
n_episodes = 70_000        # Number of episodes to practice
start_epsilon = 1.0         # Start with 100% random actions
epsilon_decay = start_epsilon / (n_episodes / 2)  # Reduce exploration over time
final_epsilon = 0.1         # Always keep some exploration

# TODO: Create environment and agent
env = ...
print(env.action_space)
print(env.observation_space)

env = gym.wrappers.RecordEpisodeStatistics(env, buffer_length=n_episodes)

# TODO: Import the agent
agent = ...

for episode in tqdm(range(n_episodes)):
    # Start a new hand
    obs, info = env.reset()
    done = False

    # Play one complete hand
    while not done:
        # Agent chooses action (initially random, gradually more intelligent)
        action = agent.get_action(obs)

        # Take action and observe result
        next_obs, reward, terminated, truncated, info = env.step(action)

        # Learn from this experience
        agent.update(obs, action, reward, terminated, next_obs)

        # Move to next state
        done = terminated or truncated
        obs = next_obs

    # Reduce exploration rate (agent becomes less random over time)
    agent.decay_epsilon()

In [ ]:
# Run the code below to test your agent!

In [ ]:
# @title
import numpy as np

def test_agent(agent, env, num_episodes=100, render_last=True):
    """Test agent performance without learning or exploration."""
    total_rewards = []
    successes = 0  # Track how many times we actually hit the target

    # Temporarily disable exploration for testing (pure exploitation)
    old_epsilon = getattr(agent, 'epsilon', 0.0)
    agent.epsilon = 0.0

    for i in range(num_episodes):
        obs, info = env.reset()
        episode_reward = 0
        done = False

        # Decide if we want to print the text render for this specific episode
        should_render = render_last and (i == num_episodes - 1)

        if should_render:
            print("\n--- Rendering Final Test Episode ---")
            env.render()

        while not done:
            # Assuming your agent has a get_action method that takes the state
            action = agent.get_action(obs)
            obs, reward, terminated, truncated, info = env.step(action)

            episode_reward += reward
            done = terminated or truncated

            if should_render:
                env.render()

            # If the episode terminated (not truncated), the agent reached the target!
            if terminated:
                successes += 1

        total_rewards.append(episode_reward)

    # Restore original epsilon
    agent.epsilon = old_epsilon

    # Calculate win rate based on successful terminations, not positive rewards
    win_rate = successes / num_episodes
    average_reward = np.mean(total_rewards)

    print(f"\nTest Results over {num_episodes} episodes:")
    print(f"Win Rate (Reached Target): {win_rate:.1%}")
    print(f"Average Reward: {average_reward:.3f}")
    print(f"Standard Deviation: {np.std(total_rewards):.3f}")


# Test your agent
env = DiscreteThermostatEnv(render_mode="human")
test_agent(agent, env, num_episodes=100, render_last=True)